# Task 2 : Retrieval & generation (RAG)
Here we define the parameters for our system:
- User query
- Vectorization strategy (SBERT, Word2Vec, TF-IDF)
- Similarity metric (Cosine, Euclidean)
- LLM information (backend, model, max tokens)


In [1]:
# User query
query = "What is a Neural Network?"

# Vectorization strategy (SBERT, Word2Vec, TF-IDF) (Note : have to match the one used in Task 1 for the embeddings)
embedding_strategy = "sbert"                # possible choices: "sbert", "word2vec", "tfidf"

# similarity metrics (Cosine, Euclidean)
similarity_strategy = "cosine"              # Possible choices: "cosine", "euclidean"

# Number of chunks to retrieve
number_of_text_chunks_to_retrieve = 4

# Number of questions to generate for the exam
number_of_questions_to_generate = 2

# tool used for generation
generator_backend = "ollama"                

# model used (Note: be sure to have download the model in Ollama beforehand)
generator_model = "llama3.2:3b"

# max tokens to send to the LLM 
max_context_tokens = 2048                   

# directories (Note: be sure to have the permissions for these folders)
input_dir = "/home/guest/shared/course_material/"
output_dir = "/home/guest/shared/course_material/output/"

print("Rag parameters charged")

Rag parameters charged


## Spark session & UDF

Instead of using .collect(), we decided to use UDFs to apply the mathematical metrics directly on the spark workers and thus accelerate the retrieval process.

In [2]:
import os
import warnings
import numpy as np
import requests
from sentence_transformers import SentenceTransformer

from pyspark.sql import SparkSession
from pyspark.sql.functions import udf, col, desc, asc
from pyspark.sql.types import FloatType

warnings.filterwarnings('ignore', category=UserWarning)

# initiate Spark session
spark = SparkSession.builder.appName("RAG_Retrieval_Engine").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
# SBERT tries to use his own parallelism which can cause issue so we disable it to have the control.
os.environ["TOKENIZERS_PARALLELISM"] = "false"

#UDFs for the metrics
def make_spark_cosine_udf(query_vector):
    """
    Create a spark udf to calculate the cosine similarity between the query vector and the document vector.
    
    Input : query vector from the user question and document vector from the database
    Output : similarity score between the two vectors (float between 0 and 1, where 1 means identical)
    
    """
    # We want this function to be executed on the spark workers
    def compute_cosine(doc_emb):
        # if a chunk is empty return 0 similarity to rank it down
        if doc_emb is None: return 0.0
        q = np.array(query_vector, dtype=np.float32)
        d = np.array(doc_emb, dtype=np.float32)
        norm_q = np.linalg.norm(q)
        norm_d = np.linalg.norm(d)
        if norm_q == 0 or norm_d == 0: return 0.0 # ensure that we don't divide by zero
        return float(np.dot(q, d) / (norm_q * norm_d))
        
    return udf(compute_cosine, FloatType()) # we pack the function in a Spark UDF and we say that it returns a float


def make_spark_euclidean_udf(query_vector):
    """
    Create a spark udf to calculate the euclidean distance between the query vector and the document vector.

    Input : query vector from the user question and document vector from the database
    Output : distance between the two vectors (float, where 0 means identical)

    """
    def compute_euclidean(doc_emb):
        if doc_emb is None: return 999.0 # In case of empty chunk, return 999 to rank it down
        q = np.array(query_vector, dtype=np.float32)
        d = np.array(doc_emb, dtype=np.float32)
        return float(np.linalg.norm(q - d)) # euclidean formula
    return udf(compute_euclidean, FloatType())

print("UDFs for similarity ready")

Using Spark's default log4j profile: org/apache/spark/log4j-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/21 20:11:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/21 20:11:57 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


UDFs for similarity ready


## Retrieval function
Transform the query in a mathematical vector using the model used in task 1. 
Then apply the udf for each document in the spark cluster.
Finally we use the DataFrame API to sort the results and use .limit(k).collect() to only bring the "Top K" in the memory of our main program.


In [3]:
def retrieve_top_chunks(query: str, embedding_strategy: str, similarity_metric: str, k: int = 4):
    """
    Retrieve the k texts that are the most similar to the query according to the metrics chosen.
    
    :param query: The question asked by the user.
    :param embedding_strategy: The method used to vectorize the documents and the query. 
    :param similarity_metric: The formula used to calculate the similarity between the query and the document vectors.
    :param k: Number of text chunks to retrieve.
    :return: A list of the k most similar text chunks with their metadata and similarity scores.
    """
    print(f"[Strategy: {embedding_strategy} | Metric: {similarity_metric}]")
    
    # Load the dataframe
    parquet_path = f"{input_dir}/embeddings_{embedding_strategy}.parquet"    
    try: # Security to not break if the parquet file is not found
        df_corpus = spark.read.parquet(parquet_path)
    except Exception as e:
        print(f"Error : No file found for {embedding_strategy}. Be sure to have run Task 1 (Notebook_1.ipynb) with the same embedding strategy. {e}")
        return []

    # Query vector
    # Important : use the same tool to vectorize the query and the documents
    if embedding_strategy == "sbert":
        encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2") # for sbert we only have to download the model
        q_vec = encoder.encode(query, normalize_embeddings=True)
    elif embedding_strategy == "word2vec":
        from pyspark.ml.feature import Word2VecModel, Tokenizer
        # For MLlib, we need to creat a dataframe with only one line with the query
        df_query = spark.createDataFrame([(query,)], ["chunk_text"])
        # tokenize the sentence in words
        tokenizer = Tokenizer(inputCol="chunk_text", outputCol="words")
        wordsData = tokenizer.transform(df_query)
        # Load word2vec model trained in Task 1.
        w2v_model = Word2VecModel.load(f"{input_dir}/w2v_model.model")
        # transform words in vectors
        w2vData = w2v_model.transform(wordsData)
        # We extract the vector from the dataframe(only one line).
        q_vec = w2vData.select("features").first()[0].toArray()
        
        
    elif embedding_strategy == "tfidf": # same logic as word2vec
        from pyspark.ml.feature import IDFModel, Tokenizer, HashingTF
        df_query = spark.createDataFrame([(query,)], ["chunk_text"])        # For MLlib, we need to creat a dataframe with only one line with the query
        tokenizer = Tokenizer(inputCol="chunk_text", outputCol="words")     # tokenize the sentence in words
        wordsData = tokenizer.transform(df_query)
        hashingTF = HashingTF(inputCol="words", outputCol="rawFeatures", numFeatures=20000) # hash trick to vectorize the words and save memory 
        featurizedData = hashingTF.transform(wordsData)
        idfModel = IDFModel.load(f"{input_dir}/tfidf_model.model")   # loading the idf model from task 1 who gives the weight of each word
        rescaledData = idfModel.transform(featurizedData)
        q_vec = rescaledData.select("features").first()[0].toArray()         # We extract the vector from the dataframe(only one line).

    
    else:
        raise ValueError(f"Unknown strategy : {embedding_strategy}, please chose between sbert, word2vec and tfidf")

    if similarity_metric == "cosine":
        metric_udf = make_spark_cosine_udf(q_vec)   # Create the udf for cosine similarity. 
        df_scored = df_corpus.withColumn("score", metric_udf(col("embedding"))) #add the column "score" to the dataframe and apply the udf
        df_top_k = df_scored.orderBy(desc("score")).limit(k)    # rank the documents and only keep the best k results        
    elif similarity_metric == "euclidean":
        metric_udf = make_spark_euclidean_udf(q_vec)    # Create the udf for euclidean distance.
        df_scored = df_corpus.withColumn("score", metric_udf(col("embedding")))  #add the column "score" to the dataframe and apply the udf
        df_top_k = df_scored.orderBy(asc("score")).limit(k) # inverse the order and keep the lowest k scores (looking for minimal distance)
    else:
        raise ValueError(f"Unknown metric : {similarity_metric}, please chose between cosine and euclidean")
    return df_top_k.collect() # we only do the collect on the top k results.



# TEST

resultats = retrieve_top_chunks(
    query=query, 
    embedding_strategy=embedding_strategy, 
    similarity_metric=similarity_strategy, 
    k=number_of_text_chunks_to_retrieve
)

for i, row in enumerate(resultats, 1):
    print(f"rank {i} [score: {row['score']:.4f}] - pdf_source: {row['source_pdf']} (p.{row['page_num']})")

[Strategy: sbert | Metric: cosine]


rank 1 [score: 0.6328] - pdf_source: a_Big Data, Data Mining, and Machine Learning ( PDFDrive.com )-min.pdf (p.117)
rank 2 [score: 0.5829] - pdf_source: A Neural Network Playground.pdf (p.1)
rank 3 [score: 0.5711] - pdf_source: a_Big Data, Data Mining, and Machine Learning ( PDFDrive.com )-min.pdf (p.117)
rank 4 [score: 0.5706] - pdf_source: Intro_DL_H515_2026.pdf (p.25)


## Conclusion on the results 
It is normal that the scores of sbert are better than the ones of word2vec and tfidf
- Sbert generates vectors with  an awarness of the global context of the sentence
- Word2Vec does an average of the words and take out the meaning of the sentence
- tfidf looks for the exact words and generate vectors with a lot of zeros 
        

## LLM and Prompt
This last step of task 2 takes the retrieved chunks and put them in a prebuild prompt to send to the LLM (ollama). The prompt is very strict to force the use of only the provided sources and to make the LLM include citations for each question and answer. We want to avoid hallucinations as much as possible.

In [4]:
def generate_exam_with_llm(query: str, spark_results, num_questions: int = 3):
    """
    Sends a prompt with the retrieved chunks to the LLM to generate an exam with questions and answers.
    
    inputs : 
    - query : query of the user 
    - spark_results : retrieved chunks 
    - num_questions : number of questions to generate
    
    """
    

    if not spark_results:   # if there's no retrieved chunks
        return "No file matched your query"




    context_text = ""
    for idx, row in enumerate(spark_results, 1):
        # create the context for the rompt
        context_text += f"\n[{idx}] (Source_pdf: {row['source_pdf']}, Page {row['page_num']}, ID: {row['chunk_id']})\n{row['chunk_text']}\n"
    
    # Constructing the Prompt
    prompt = f"""You are a strict university teaching assistant designing an exam.
TASK: Generate {num_questions} exam questions about the topic: "{query}".

RULES:
- Base your questions and answers EXCLUSIVELY on the provided SOURCES.
- If the sources do not contain enough information, state: "Insufficient information in the provided sources."
- You MUST append citations at the end of each question and each answer.

OUTPUT FORMAT:
Q1. [Question text here] [Source_pdf,Page,ID]
A1. [Answer text here] [Source_pdf,Page,ID]

SOURCES:
{context_text}
"""
    
    # 3. Affichage pour le professeur (Deliverable 3 : Prouver les paramètres envoyés)
    print(" Generation parameters used :")
    print(f"Backend : {generator_backend}")
    print(f"Model : {generator_model}")
    print(f"Max context tokens : {max_context_tokens}")
    print("Prompt sent to the LLM :")
    print(prompt)

    try:
        # request to the LLM API
        response = requests.post(
            "http://host.docker.internal:11434/api/generate",
            json={
                "model": "llama3.2:3b", # be sure to have downloaded the model in Ollama, otherwise it will raise an error
                "prompt": prompt,
                "stream": False # waits for the answer to be completely done before sending it back.
            },
            timeout=120 # 2 minutes timeout avoiding long waits
        )
        response.raise_for_status() # error if connexion failed        
        return response.json()["response"] 
    except requests.exceptions.RequestException as e:
        return f"Error with the connexion with the api. {e}"



# Final generation and save the output

examen_genere = generate_exam_with_llm(
    query=query, 
    spark_results=resultats, 
    num_questions=number_of_questions_to_generate
)

print("EXAM: \n")
print(examen_genere)


# make sure we create the output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# output file path
fichier_sortie = f"{output_dir}/examen_genere_{query}_{embedding_strategy}_{similarity_strategy}.txt"

# 3. On écrit l'examen dans le fichier
with open(fichier_sortie, "w", encoding="utf-8") as f:
    f.write(f"Subject : {query}\n")
    f.write(examen_genere)

print(f"Exam has been saved in :\n{fichier_sortie}")

 Generation parameters used :
Backend : ollama
Model : llama3.2:3b
Max context tokens : 2048
Prompt sent to the LLM :
You are a strict university teaching assistant designing an exam.
TASK: Generate 2 exam questions about the topic: "What is a Neural Network?".

RULES:
- Base your questions and answers EXCLUSIVELY on the provided SOURCES.
- If the sources do not contain enough information, state: "Insufficient information in the provided sources."
- You MUST append citations at the end of each question and each answer.

OUTPUT FORMAT:
Q1. [Question text here] [Source_pdf,Page,ID]
A1. [Answer text here] [Source_pdf,Page,ID]

SOURCES:

[1] (Source_pdf: a_Big Data, Data Mining, and Machine Learning ( PDFDrive.com )-min.pdf, Page 117, ID: p117_c1)
activation function of the hidden layer neurons vector Z = weights between the hidden layer neurons and the output layer neurons func tion o= activation function of the output layer neurons Networks like these, generalized to include multi ple hi